In [13]:
from transformers import NllbTokenizer
from optimum.onnxruntime import ORTModelForSeq2SeqLM
import torch

In [14]:
model_path = "assets/nllb-distilled"

tokenizer = NllbTokenizer.from_pretrained(model_path, local_files_only=True)
model = ORTModelForSeq2SeqLM.from_pretrained(model_path, local_files_only=True)

def translate(text, src_lang, tgt_lang):
    # 소스 언어 설정
    tokenizer.src_lang = src_lang

    # 입력 인코딩
    encoded = tokenizer(text, return_tensors="pt")

    # 타깃 언어의 BOS 토큰 ID를 직접 변환해서 사용
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)

    with torch.no_grad():
        generated = model.generate(**encoded, forced_bos_token_id=forced_bos_token_id)

    return tokenizer.decode(generated[0], skip_special_tokens=True)

Could not find any ONNX files with standard file name decoder_model_merged.onnx, files found: [WindowsPath('decoder_model.onnx'), WindowsPath('decoder_with_past_model.onnx'), WindowsPath('encoder_model.onnx')]. Please make sure to pass a `file_name` and/or `subfolder` argument to `from_pretrained` when loading an ONNX file with non-standard file names.


In [15]:
# 번역 테스트
print("영어")
ko_to_en = translate("안녕하세요, 오늘 날씨가 참 좋네요.", "kor_Hang", "eng_Latn")
en_to_ko = translate("Hello, the weather is really nice today.", "eng_Latn", "kor_Hang")
print("🇰🇷 → 🇺🇸:", ko_to_en)
print("🇺🇸 → 🇰🇷:", en_to_ko)

print("\n일본어")
ko_to_ja = translate("안녕하세요, 오늘 날씨가 참 좋네요.", "kor_Hang", "jpn_Jpan")
ja_to_ko = translate("こんにちは、今日はとてもいい天気ですね。", "jpn_Jpan", "kor_Hang")
print("🇰🇷 → 🇯🇵:", ko_to_ja)
print("🇯🇵 → 🇰🇷:", ja_to_ko)

print("\n스페인어")
ko_to_es = translate("안녕하세요, 오늘 날씨가 참 좋네요.", "kor_Hang", "spa_Latn")
es_to_ko = translate("Hola, el clima está muy agradable hoy.", "spa_Latn", "kor_Hang")
print("🇰🇷 → 🇪🇸:", ko_to_es)
print("🇪🇸 → 🇰🇷:", es_to_ko)

print("\n러시아어")
ko_to_ru = translate("안녕하세요, 오늘 날씨가 참 좋네요.", "kor_Hang", "rus_Cyrl")
ru_to_ko = translate("Здравствуйте, сегодня очень хорошая погода.", "rus_Cyrl", "kor_Hang")
print("🇰🇷 → 🇷🇺:", ko_to_ru)
print("🇷🇺 → 🇰🇷:", ru_to_ko)

print("\n중국어 - 중국대륙")
ko_to_zh_hans = translate("안녕하세요, 오늘 날씨가 참 좋네요.", "kor_Hang", "zho_Hans")
zh_hans_to_ko = translate("你好，今天天气真好。", "zho_Hans", "kor_Hang")
print("🇰🇷 → 🇨🇳(간체):", ko_to_zh_hans)
print("🇨🇳(간체) → 🇰🇷:", zh_hans_to_ko)

print("\n중국어 - 대만, 홍콩, 마카오")
ko_to_zh_hant = translate("안녕하세요, 오늘 날씨가 참 좋네요.", "kor_Hang", "zho_Hant")
zh_hant_to_ko = translate("你好，今天天氣真好。", "zho_Hant", "kor_Hang")
print("🇨🇳(번체) → 🇰🇷:", ko_to_zh_hant)
print("🇰🇷 → 🇨🇳(번체):", zh_hant_to_ko)

영어
🇰🇷 → 🇺🇸: Hello, the weather is very nice today.
🇺🇸 → 🇰🇷: 안녕하세요, 오늘 날씨가 정말 좋네요.

일본어
🇰🇷 → 🇯🇵: こんにちは 今日は天気が良いです
🇯🇵 → 🇰🇷: 안녕하세요, 날씨가 아주 좋네요.

스페인어
🇰🇷 → 🇪🇸: Hola, el tiempo es muy bueno.
🇪🇸 → 🇰🇷: 안녕하세요, 날씨는 매우 좋았습니다.

러시아어
🇰🇷 → 🇷🇺: Привет, сегодня очень хорошая погода.
🇷🇺 → 🇰🇷: 안녕하세요, 오늘 날씨가 아주 좋네요.

중국어 - 중국대륙
🇰🇷 → 🇨🇳(간체): 你好,今天天气很好.
🇨🇳(간체) → 🇰🇷: 안녕하세요, 날씨가 좋네요.

중국어 - 대만, 홍콩, 마카오
🇨🇳(번체) → 🇰🇷: 您好,今天天天氣非常好.
🇰🇷 → 🇨🇳(번체): 안녕하세요, 날씨가 좋네요.


In [18]:
# 토크나이저가 지원하는 언어 코드 리스트 출력
print("zho_Hant →", tokenizer.convert_tokens_to_ids("zho_Hant"))

zho_Hant → 256201


In [23]:
def new_translate(text, src_lang, tgt_lang):
    # 소스 언어 설정
    tokenizer.src_lang = src_lang

    # 입력 문장을 토큰화
    encoded = tokenizer(text, return_tensors="pt")

    # 타깃 언어 BOS 토큰 ID
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)

    # 번역 생성 (출력 길이 늘리기)
    with torch.no_grad():
        generated_tokens = model.generate(
            **encoded,
            forced_bos_token_id=forced_bos_token_id,
            max_length=128,          # 충분히 긴 출력
            num_beams=5,             # 빔 서치로 품질 향상
            no_repeat_ngram_size=2,  # 같은 n-gram 반복 방지
            repetition_penalty=1.2,  # 반복 억제
            early_stopping=True      # 불필요한 반복 방지
        )

    return tokenizer.decode(generated_tokens[0], skip_special_tokens=True)

In [24]:
# 재번역 테스트
print("중국어 - 대만, 홍콩, 마카오")
ko_to_zh_hant = new_translate("안녕하세요, 오늘 날씨가 참 좋네요.", "kor_Hang", "zho_Hant")
zh_hant_to_ko = new_translate("你好，今天天氣真好。", "zho_Hant", "kor_Hang")
print("🇨🇳(번체) → 🇰🇷:", ko_to_zh_hant)
print("🇰🇷 → 🇨🇳(번체):", zh_hant_to_ko)

중국어 - 대만, 홍콩, 마카오
🇨🇳(번체) → 🇰🇷: 您好,今天的天氣非常好.
🇰🇷 → 🇨🇳(번체): 안녕하세요, 오늘 날씨가 좋네요.


In [ ]:
# 중국어 - 간체, 번체 테스트
test_sentences = [
    "안녕하세요, 오늘 날씨가 참 좋네요.",
    "저는 한국에서 왔습니다.",
    "이 책은 정말 재미있습니다.",
    "커피 한 잔 마시고 싶어요.",
    "내일은 친구와 영화를 볼 예정입니다."
]

for text in test_sentences:
    print("\n원문:", text)

    # 간체 번역
    zh_hans = new_translate(text, "kor_Hang", "zho_Hans")
    print("🇨🇳 간체:", zh_hans)

    # 번체 번역
    zh_hant = new_translate(text, "kor_Hang", "zho_Hant")
    print("🇨🇳 번체:", zh_hant)


원문: 안녕하세요, 오늘 날씨가 참 좋네요.
🇨🇳 간체: 你好,今天天气很好.
🇨🇳 번체: 您好,今天的天氣非常好.

원문: 저는 한국에서 왔습니다.
🇨🇳 간체: 我来自韩国.
🇨🇳 번체: 我來自韓國.

원문: 이 책은 정말 재미있습니다.
🇨🇳 간체: 这本书非常有趣.
🇨🇳 번체: 這本書非常有趣.

원문: 커피 한 잔 마시고 싶어요.
🇨🇳 간체: 我想喝一杯咖啡.
🇨🇳 번체: 我想要喝一杯咖啡.

원문: 내일은 친구와 영화를 볼 예정입니다.
🇨🇳 간체: 明天我和朋友一起看电影.
🇨🇳 번체: 明天我們和朋友一起看電影.
